#Actividad 1: Estructura de un NLU

1. Instalación, importación y cargar modelo.




In [1]:

# Instalación de dependencias
!pip install gradio spacy scikit-learn
!python -m spacy download es_core_news_sm

# Importar librerías
import gradio as gr
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import re

# Cargar modelo NLP en español
nlp_es = spacy.load("es_core_news_sm")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.2
    Uninstalling MarkupSafe-3.0.2:
      Successfully uninstalled MarkupSafe-3.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
tor

2. Declaramos las intenciones y enunciados de muestra




In [2]:
intents_data = {
    "consultar_saldo": {
        "examples": [
            "¿Cuál es mi saldo?",
            "Quiero saber mi saldo",
            "Necesito conocer mi saldo"
        ],
        "response": "Tu saldo actual es de 1,000 euros."
    },
    "transferir_dinero": {
        "examples": [
            "Quiero transferir 500 euros a mi amigo",
            "Transfiere 200 euros a la cuenta de Juan",
            "Necesito enviar dinero a mi madre"
        ],
        "response": "La transferencia ha sido realizada con éxito."
    },
    "pagar_factura": {
        "examples": [
            "Quiero pagar la factura del agua",
            "¿Puedo pagar mi recibo de electricidad?",
            "Necesito pagar mi factura"
        ],
        "response": "Tu factura ha sido pagada correctamente."
    },
    "hablar_con_agente": {
        "examples": [
            "Quiero hablar con un agente",
            "Necesito ayuda de un agente",
            "¿Puedo comunicarme con soporte?"
        ],
        "response": "Conectándote con un agente, por favor espera..."
    }
}

3. Preparación, vectorización y entrenamiento.

In [3]:
texts = []
labels = []
for intent, data in intents_data.items():
    texts.extend(data["examples"])
    labels.extend([intent] * len(data["examples"]))

# Vectorización de texto
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

# Entrenar clasificador
classifier = LogisticRegression()
classifier.fit(X, labels)

LogisticRegression()

4. Función para predecir intención

In [4]:
def predict_intent(text):
    X_test = vectorizer.transform([text])
    predicted_intent = classifier.predict(X_test)[0]
    confidence = max(classifier.predict_proba(X_test)[0])
    return predicted_intent, confidence

5. Función para procesar texto y extraer entidades y slots personalizados

In [5]:


def extract_entities_and_slots(text):
    # Extraer entidades con spaCy
    doc = nlp_es(text)
    entities = {ent.text: ent.label_ for ent in doc.ents if ent.label_ in ["LOC", "DATE", "ORG", "PER", "MONEY"]}

    # Detectar montos y cuentas en texto
    amount_match = re.search(r"\b(\d+\s?euros)\b", text, re.IGNORECASE)
    if amount_match:
        entities["Monto"] = amount_match.group(1)

    account_match = re.search(r"cuenta\s(de\s)?([A-ZÁÉÍÓÚa-záéíóú]+)", text, re.IGNORECASE)
    if account_match:
        entities["Cuenta"] = account_match.group(2)

    return entities

def language_understanding(text):
    # Predecir intención
    intent, confidence = predict_intent(text)

    # Extraer entidades y slots
    entities = extract_entities_and_slots(text)

    # Generar respuesta según intención
    if intent in intents_data:
        response = intents_data[intent]["response"]
        if "Monto" in entities:
            response += f" He detectado un monto de {entities['Monto']}."
        if "Cuenta" in entities:
            response += f" He detectado que quieres usar la cuenta de {entities['Cuenta']}."
    else:
        response = "Lo siento, no entendí eso. ¿Puedes reformularlo?"

    return {"Respuesta": response, "Intención": intent, "Confianza": f"{confidence:.2f}", "Entidades": entities}

# 7️⃣ Crear interfaz de chat con Gradio
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="Chatbot Bancario en Español", type="messages")
    with gr.Row():
        txt = gr.Textbox(show_label=False, placeholder="Escribe tu mensaje aquí... (en español)")
        btn = gr.Button("Enviar")

    def chat_interface(history, user_input):
        result = language_understanding(user_input)
        response = result["Respuesta"]
        # Mostrar el resultado completo (intención, confianza y entidades) en el chat
        debug_info = f"\nIntención: {result['Intención']}\nConfianza: {result['Confianza']}\nEntidades: {result['Entidades']}"
        history.append({"role": "user", "content": user_input})
        history.append({"role": "assistant", "content": response + debug_info})
        return history, ""

    btn.click(chat_interface, [chatbot, txt], [chatbot, txt])

demo.launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6cbac12857a334b66b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
